In [1]:
# ==============================================================================
# Binary Time Series Deep Learning using LSTM with Multi-Head Attention
#
# Project: Network Intrusion Detection on CIC-IDS2018
# Author: B12 - BPPIMT - CSE 21-25
# Date: 12-06-25
#
# Description:
#   This script trains a robust, binary classifier (Benign vs. Attack) using
#   a data-centric approach. It first uses a RandomForest to perform feature
#   selection, identifying the most impactful features. The final LSTM+Attention
#   model is then trained on this cleaner, high-signal subset of the data
#   for optimal real-world performance.
# ==============================================================================

# ==============================================================================
# 1. SETUP: LIBRARIES AND ENVIRONMENT
# ==============================================================================
import os
import gc
import glob
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. GLOBAL CONFIGURATION & HYPERPARAMETERS
# ==============================================================================
DATA_PATH = '/kaggle/input/cse-cic-ids2018/CSE-CIC-IDS2018/'
EXPORT_PATH = './final_tuned_model/'
RANDOM_STATE = 42

# --- Feature Selection Parameter ---
NUM_FEATURES_TO_SELECT = 40

# --- Data & Time-Series Parameters ---
TEST_SIZE = 0.2
WINDOW_LENGTH = 40
STRIDE = 20

# --- "Sweet Spot" Model Architecture ---
LSTM_UNITS = 64
ATTENTION_HEADS = 4
ATTENTION_KEY_DIM = 32
DROPOUT_RATE = 0.4
DENSE_UNITS = 32

# --- Training Parameters ---
BATCH_SIZE = 1024
EPOCHS = 50
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 12 # Slightly increased patience for smoother training
LR_SCHEDULER_PATIENCE = 5

os.makedirs(EXPORT_PATH, exist_ok=True)
print(f"All model artifacts will be saved to: {EXPORT_PATH}")

# ==============================================================================
# 3. DATA LOADING AND FEATURE SELECTION
# ==============================================================================
print("\n[INFO] Step 3: Loading data and performing feature selection...")
try:
    all_files = glob.glob(os.path.join(DATA_PATH, "*.csv"))
    df = pd.concat((pd.read_csv(f, encoding='latin1', low_memory=False) for f in all_files), ignore_index=True)
except Exception as e:
    print(f"[ERROR] Data loading failed: {e}"); exit()

# --- Initial Cleaning ---
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
COLS_TO_DROP = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp', 'Protocol']
df.drop(columns=COLS_TO_DROP, errors='ignore', inplace=True)
feature_cols = [col for col in df.columns if col != 'Label']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.dropna(inplace=True)
df['Label'] = np.where(df['Label'] == 'Benign', 0, 1) # 0 for BENIGN, 1 for ATTACK
print(f"Initial data loaded. Shape: {df.shape}")

# --- Feature Selection using a RandomForest on a balanced sample ---
print(f"\nSelecting top {NUM_FEATURES_TO_SELECT} features for model training...")
sampler = RandomUnderSampler(sampling_strategy={0: 50000, 1: 50000}, random_state=RANDOM_STATE)
X_sample, y_sample = sampler.fit_resample(df.drop(columns=['Label']), df['Label'])

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({'feature': X_sample.columns, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values('importance', ascending=False)
TOP_FEATURES = feature_importance_df.head(NUM_FEATURES_TO_SELECT)['feature'].tolist()

print(f"Top {NUM_FEATURES_TO_SELECT} features identified:")
print(", ".join(TOP_FEATURES))

# Filter the original dataframe to keep only the most important features
df = df[TOP_FEATURES + ['Label']]
print(f"\nDataframe filtered. New shape: {df.shape}")
del X_sample, y_sample, rf, importances, feature_importance_df; gc.collect()

# ==============================================================================
# 4. FINAL DATA PREPARATION AND SPLITTING
# ==============================================================================
print("\n[INFO] Step 4: Preparing final data with selected features...")
X = df.drop(columns=['Label'])
y = df['Label'].values
del df; gc.collect()

feature_names = X.columns.tolist()
with open(os.path.join(EXPORT_PATH, 'feature_mapping.json'), 'w') as f: json.dump(feature_names, f)

label_encoder = LabelEncoder().fit(['BENIGN', 'ATTACK'])
joblib.dump(label_encoder, os.path.join(EXPORT_PATH, 'label_encoder.pkl'))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, os.path.join(EXPORT_PATH, 'scaler.pkl'))
del X; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
del X_scaled, y; gc.collect()

# ==============================================================================
# 5. DATA BALANCING VIA UNDER-SAMPLING
# ==============================================================================
print("\n[INFO] Step 5: Enforcing a 1:1 balance via RandomUnderSampler...")
TARGET_COUNT = 300000
sampling_strategy = {0: TARGET_COUNT, 1: TARGET_COUNT}
under_sampler = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE)
X_train_res, y_train_res = under_sampler.fit_resample(X_train, y_train)
print(f"Resampling complete. New training shape: {X_train_res.shape}")
del X_train, y_train; gc.collect()

# ==============================================================================
# 6. SLIDING WINDOW PREPARATION FOR TEST SET
# ==============================================================================
print("\n[INFO] Step 6: Creating sliding window sequences for the test set...")
def create_sliding_windows(X, y, window_length, stride):
    X_windows, y_windows = [], []
    for i in range(0, len(X) - window_length + 1, stride):
        X_windows.append(X[i:i + window_length])
        y_windows.append(y[i + window_length - 1])
    return np.array(X_windows), np.array(y_windows)

X_test_windows, y_test_windows = create_sliding_windows(X_test, y_test, WINDOW_LENGTH, STRIDE)
print(f"Test windows created successfully. Shape (X, y): {X_test_windows.shape}, {y_test_windows.shape}")
del X_test, y_test; gc.collect()

# ==============================================================================
# 7. MODEL ARCHITECTURE DEFINITION
# ==============================================================================
print("\n[INFO] Step 7: Building the fine-tuned LSTM + Attention model...")

def build_binary_model(input_shape):
    inputs = Input(shape=input_shape)
    # Using L2 regularization to penalize large weights and prevent overfitting
    lstm_out = LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(1e-5))(inputs)
    lstm_out = Dropout(DROPOUT_RATE)(lstm_out)
    attention_out = MultiHeadAttention(num_heads=ATTENTION_HEADS, key_dim=ATTENTION_KEY_DIM)(query=lstm_out, value=lstm_out, key=lstm_out)
    pooled_out = GlobalAveragePooling1D()(attention_out)
    pooled_out = Dropout(DROPOUT_RATE)(pooled_out)
    dense_out = Dense(DENSE_UNITS, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-5))(pooled_out)
    outputs = Dense(1, activation='sigmoid')(dense_out)
    model = Model(inputs=inputs, outputs=outputs)
    return model

input_shape = (WINDOW_LENGTH, X_train_res.shape[1])
model = build_binary_model(input_shape)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

# ==============================================================================
# 8. MODEL TRAINING
# ==============================================================================
print("\n[INFO] Step 8: Setting up data generators and starting model training...")

def training_data_generator(X_data, y_data, window_length, stride, batch_size):
    possible_start_indices = np.arange(0, X_data.shape[0] - window_length + 1, stride)
    while True:
        np.random.shuffle(possible_start_indices)
        for offset in range(0, len(possible_start_indices), batch_size):
            batch_start_indices = possible_start_indices[offset:offset + batch_size]
            X_batch = np.zeros((len(batch_start_indices), window_length, X_data.shape[1]))
            y_batch = np.zeros(len(batch_start_indices))
            for i, start_idx in enumerate(batch_start_indices):
                end_idx = start_idx + window_length
                X_batch[i] = X_data[start_idx:end_idx]
                y_batch[i] = y_data[end_idx - 1]
            yield X_batch, y_batch

def validation_data_generator(X_windowed, y_windowed, batch_size):
    num_samples = X_windowed.shape[0]
    while True:
        indices = np.arange(num_samples)
        for offset in range(0, num_samples, batch_size):
            batch_indices = indices[offset:offset + batch_size]
            yield X_windowed[batch_indices], y_windowed[batch_indices]

train_generator = training_data_generator(X_train_res, y_train_res, WINDOW_LENGTH, STRIDE, BATCH_SIZE)
validation_generator = validation_data_generator(X_test_windows, y_test_windows, BATCH_SIZE)

num_train_windows = len(np.arange(0, len(X_train_res) - WINDOW_LENGTH + 1, STRIDE))
steps_per_epoch = max(1, num_train_windows // BATCH_SIZE)
validation_steps = max(1, len(X_test_windows) // BATCH_SIZE)

# --- Callbacks for Stable, Loss-Based Training ---
monitor_metric = 'val_loss'
checkpoint_path = os.path.join(EXPORT_PATH, 'best_model.h5')
model_checkpoint = ModelCheckpoint(filepath=checkpoint_path, monitor=monitor_metric, mode='min', save_best_only=True, verbose=1)
early_stopping = EarlyStopping(monitor=monitor_metric, mode='min', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor=monitor_metric, mode='min', factor=0.2, patience=LR_SCHEDULER_PATIENCE, min_lr=1e-6, verbose=1)

print(f"Callbacks will monitor '{monitor_metric}' to find the best general-purpose model.")

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=[model_checkpoint, early_stopping, lr_scheduler],
    verbose=1
)
joblib.dump(history.history, os.path.join(EXPORT_PATH, 'training_history.pkl'))
del X_train_res, y_train_res; gc.collect()

# ==============================================================================
# 9. PERFORMANCE EVALUATION
# ==============================================================================
print("\n[INFO] Step 9: Evaluating final model performance...")
model.load_weights(checkpoint_path)

# A standard 0.5 threshold is used for baseline evaluation.
# This can be tuned later to trade off between precision and recall.
PREDICTION_THRESHOLD = 0.5
print(f"Using a standard prediction threshold of {PREDICTION_THRESHOLD}.")

y_pred_probs = model.predict(X_test_windows, batch_size=BATCH_SIZE)
y_pred = (y_pred_probs > PREDICTION_THRESHOLD).astype(int)

# --- Generate and Save Reports ---
report = classification_report(y_test_windows, y_pred, target_names=label_encoder.classes_, output_dict=True)
print("\nClassification Report:")
print(pd.DataFrame(report).transpose())
with open(os.path.join(EXPORT_PATH, 'classification_report.json'), 'w') as f: json.dump(report, f, indent=4)

cm = confusion_matrix(y_test_windows, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('Actual Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.savefig(os.path.join(EXPORT_PATH, 'confusion_matrix.png')); plt.close()

# --- Training History Visualization ---
plt.figure(figsize=(12, 6)); plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training & Validation Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training & Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(EXPORT_PATH, 'training_history_plots.png')); plt.close()

# ==============================================================================
# 10. EXPORT MODEL AND ARTIFACTS
# ==============================================================================
print("\n[INFO] Step 10: Exporting final model and configuration artifacts...")
model.save(os.path.join(EXPORT_PATH, 'intrusion_detection_model.keras'))

config = {
    "data_path": DATA_PATH,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "window_length": WINDOW_LENGTH,
    "stride": STRIDE,
    "num_features_selected": NUM_FEATURES_TO_SELECT,
    "top_features": TOP_FEATURES,
    "lstm_units": LSTM_UNITS,
    "attention_heads": ATTENTION_HEADS,
    "attention_key_dim": ATTENTION_KEY_DIM,
    "dropout_rate": DROPOUT_RATE,
    "dense_units": DENSE_UNITS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "prediction_threshold": PREDICTION_THRESHOLD,
    "label_classes": list(label_encoder.classes_)
}
with open(os.path.join(EXPORT_PATH, 'model_config.json'), 'w') as f:
    json.dump(config, f, indent=4)
print("Model configuration saved.")

try:
    from datetime import datetime
    import pkg_resources
    requirements_path = os.path.join(EXPORT_PATH, 'requirements.txt')
    installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    with open(requirements_path, 'w') as f:
        f.write(f"# Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        for package, version in sorted(installed_packages.items()):
            f.write(f"{package}=={version}\n")
    print("Environment requirements saved.")
except Exception as e:
    print(f"[WARNING] Could not generate requirements.txt: {e}")

print("\n[SUCCESS] Script execution complete.")

TensorFlow Version: 2.16.1
All model artifacts will be saved to: ./final_tuned_model/

[INFO] Step 3: Loading data and performing feature selection...
Initial data loaded. Shape: (9625148, 77)

Selecting top 40 features for model training...
Top 40 features identified:
Subflow Fwd Byts, Init Fwd Win Byts, Init Bwd Win Byts, TotLen Fwd Pkts, Fwd Pkt Len Max, Fwd Seg Size Min, Fwd Seg Size Avg, Fwd IAT Tot, Flow Duration, Flow IAT Mean, Flow IAT Min, Fwd Header Len, Fwd Pkt Len Mean, Fwd Pkts/s, Fwd IAT Max, Fwd IAT Mean, Tot Fwd Pkts, Flow IAT Max, Pkt Size Avg, Flow Byts/s, Bwd Pkts/s, Flow Pkts/s, Pkt Len Var, Fwd IAT Std, Subflow Fwd Pkts, Pkt Len Max, Fwd IAT Min, Subflow Bwd Pkts, Bwd IAT Min, Bwd Pkt Len Mean, Down/Up Ratio, Bwd Seg Size Avg, Pkt Len Std, Pkt Len Mean, Tot Bwd Pkts, ECE Flag Cnt, Flow IAT Std, Bwd Pkt Len Max, Fwd Pkt Len Std, Bwd Header Len

Dataframe filtered. New shape: (9625148, 41)

[INFO] Step 4: Preparing final data with selected features...

[INFO] Step 5:

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 40, 40)         │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ (None, 40, 64)         │         26,880 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 40, 64)         │              0 │ lstm[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 40, 64)         │         33,216 │ dropout[0][0],         │
│ (MultiHeadAttention)      │                        │                │ dropout[0][0],         │
│                           │                        │                │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d  │ (None, 64)             │              0 │ multi_head_attention[… │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 64)             │              0 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 32)             │          2,080 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 1)              │             33 │ dense[0][0]            │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 62,209 (243.00 KB)

 Trainable params: 62,209 (243.00 KB)

 Non-trainable params: 0 (0.00 B)


[INFO] Step 8: Setting up data generators and starting model training...
Callbacks will monitor 'val_loss' to find the best general-purpose model.
Epoch 1/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - accuracy: 0.8958 - loss: 0.4576 - precision: 0.8938 - recall: 0.8845
Epoch 1: val_loss improved from inf to 2.84132, saving model to ./final_tuned_model/best_model.h5
29/29 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.8979 - loss: 0.4504 - precision: 0.8957 - recall: 0.8873 - val_accuracy: 0.6876 - val_loss: 2.8413 - val_precision: 0.2903 - val_recall: 0.0687 - learning_rate: 0.0010
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 546ms/step - accuracy: 1.0000 - loss: 0.0013 - precision: 1.0000 - recall: 0.9999
Epoch 2: val_loss did not improve from 2.84132
29/29 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 1.0000 - loss: 0.0014 - precision: 1.0000 - recall: 0.9999 - val_accuracy: 0.7062 - val_loss: 5.0449 - val_precision: 0.2866 - val_recall: 0.0234 - learning_rate: 0.0010
Epoch 3/50
29/29 ━━